In [17]:
import math

import numpy as np
import pandas as pd
import openai

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, SparseVectorParams, Modifier, PayloadSchemaType, Document

In [3]:
COLLECTION_NAME = "Recipes-collection-01-hybrid"

In [4]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [14]:
qdrant_client.create_collection(
  collection_name=COLLECTION_NAME,
  vectors_config={
    "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
  },
  sparse_vectors_config={
    "bm25": SparseVectorParams(modifier=Modifier.IDF)
  }
)

True

In [15]:
col_indexes = [
  { "name": "Calories", "type": PayloadSchemaType.FLOAT },
  { "name": "ProteinContent", "type": PayloadSchemaType.FLOAT },
  { "name": "CarbohydrateContent", "type": PayloadSchemaType.FLOAT },
  { "name": "FatContent", "type": PayloadSchemaType.FLOAT },
  { "name": "total_time_minutes", "type": PayloadSchemaType.INTEGER },
]

for c in col_indexes:
  qdrant_client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name=c["name"],
    field_schema=c["type"]
  )

In [9]:
def get_embeddings(texts, model="text-embedding-3-small"):
  response = openai.embeddings.create(
    input=texts,
    model=model
  )
  # sort by .index so embeddings stay aligned with the input order
  return [d.embedding for d in sorted(response.data, key=lambda d: d.index)]


def get_embedding(text, model="text-embedding-3-small"):
  # single-text helper (handy later for embedding a query)
  return get_embeddings([text], model=model)[0]

In [10]:
df_items = pd.read_parquet("../data/recipes_sample.parquet")

In [12]:
data_to_embed = df_items.to_dict(orient="records")

In [20]:
BATCH_SIZE = 100


def clean(v):
  # Make a value JSON-serializable for a Qdrant payload.
  if isinstance(v, np.ndarray):          # Images, Keywords, ingredients, instructions...
    return [clean(x) for x in v.tolist()]
  if isinstance(v, list):
    return [clean(x) for x in v]
  if isinstance(v, pd.Timestamp):        # DatePublished -> ISO string
    return None if pd.isna(v) else v.isoformat()
  if isinstance(v, np.generic):          # numpy scalar -> native python
    v = v.item()
  if v is None or (isinstance(v, float) and math.isnan(v)):  # NaN/NaT -> None
    return None
  return v


def to_payload(row):
  return {k: clean(v) for k, v in row.items() if k != "text"}


pointstructs = []
for start in range(0, len(data_to_embed), BATCH_SIZE):
  batch = data_to_embed[start:start + BATCH_SIZE]
  embeddings = get_embeddings([row["text"] for row in batch])
  for row, embedding in zip(batch, embeddings):
    pointstructs.append(
      PointStruct(
        id=int(row["RecipeId"]),
        vector={
          "text-embedding-3-small": embedding,
          "bm25": Document(
            text=row["text"],
            model="Qdrant/bm25",
          )
        },
        payload=to_payload(row),
      )
    )
  print(f"embedded {min(start + BATCH_SIZE, len(data_to_embed)):,} / {len(data_to_embed):,}")

embedded 100 / 4,983
embedded 200 / 4,983
embedded 300 / 4,983
embedded 400 / 4,983
embedded 500 / 4,983
embedded 600 / 4,983
embedded 700 / 4,983
embedded 800 / 4,983
embedded 900 / 4,983
embedded 1,000 / 4,983
embedded 1,100 / 4,983
embedded 1,200 / 4,983
embedded 1,300 / 4,983
embedded 1,400 / 4,983
embedded 1,500 / 4,983
embedded 1,600 / 4,983
embedded 1,700 / 4,983
embedded 1,800 / 4,983
embedded 1,900 / 4,983
embedded 2,000 / 4,983
embedded 2,100 / 4,983
embedded 2,200 / 4,983
embedded 2,300 / 4,983
embedded 2,400 / 4,983
embedded 2,500 / 4,983
embedded 2,600 / 4,983
embedded 2,700 / 4,983
embedded 2,800 / 4,983
embedded 2,900 / 4,983
embedded 3,000 / 4,983
embedded 3,100 / 4,983
embedded 3,200 / 4,983
embedded 3,300 / 4,983
embedded 3,400 / 4,983
embedded 3,500 / 4,983
embedded 3,600 / 4,983
embedded 3,700 / 4,983
embedded 3,800 / 4,983
embedded 3,900 / 4,983
embedded 4,000 / 4,983
embedded 4,100 / 4,983
embedded 4,200 / 4,983
embedded 4,300 / 4,983
embedded 4,400 / 4,983
embedd

In [21]:
UPLOAD_BATCH = 64
for start in range(0, len(pointstructs), UPLOAD_BATCH):
  qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=pointstructs[start:start + UPLOAD_BATCH],
    wait=True,
  )
  print(f"upserted {min(start + UPLOAD_BATCH, len(pointstructs)):,} / {len(pointstructs):,}")

upserted 64 / 4,983
upserted 128 / 4,983
upserted 192 / 4,983
upserted 256 / 4,983
upserted 320 / 4,983
upserted 384 / 4,983
upserted 448 / 4,983
upserted 512 / 4,983
upserted 576 / 4,983
upserted 640 / 4,983
upserted 704 / 4,983
upserted 768 / 4,983
upserted 832 / 4,983
upserted 896 / 4,983
upserted 960 / 4,983
upserted 1,024 / 4,983
upserted 1,088 / 4,983
upserted 1,152 / 4,983
upserted 1,216 / 4,983
upserted 1,280 / 4,983
upserted 1,344 / 4,983
upserted 1,408 / 4,983
upserted 1,472 / 4,983
upserted 1,536 / 4,983
upserted 1,600 / 4,983
upserted 1,664 / 4,983
upserted 1,728 / 4,983
upserted 1,792 / 4,983
upserted 1,856 / 4,983
upserted 1,920 / 4,983
upserted 1,984 / 4,983
upserted 2,048 / 4,983
upserted 2,112 / 4,983
upserted 2,176 / 4,983
upserted 2,240 / 4,983
upserted 2,304 / 4,983
upserted 2,368 / 4,983
upserted 2,432 / 4,983
upserted 2,496 / 4,983
upserted 2,560 / 4,983
upserted 2,624 / 4,983
upserted 2,688 / 4,983
upserted 2,752 / 4,983
upserted 2,816 / 4,983
upserted 2,880 / 4,